# Placement and checkpoint pilot v3

Local CPU simulation only. No training tensors, Ray, GPUs, cloud instances, Z3 or learned model. Read `README.md` and `MODEL.md` for the exact assumptions.

V3 lets the planner choose **checkpoint interval as well as machine placement and extra backups**. It tests one, two and four replica groups, so existing replicas are not always the dominant recovery mechanism. Fixed-work completion time under failures remains the objective; costs and deadline success are constraints.

Restart the kernel after pulling. Run setup/settings, inspect cases, then Run/resume. Each invocation pauses after 30 minutes and can be resumed. Plotting and export do not run simulations. Use the fresh `pilot_v3` directory; preserve v1/v2 results.


In [ ]:
from pathlib import Path
import json
import sys

# Works when Jupyter starts in the repository root or notebooks/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "learnedplacement" / "runner.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the LearnedPlacement checkout.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from learnedplacement.runner import run, status, read_json, paired_report, export_run
from learnedplacement.model import make_cases
print("Repository:", ROOT)
print("Python:", sys.version.split()[0])


## Editable settings

48 cases = 3 sizes × 4 regimes × 2 instances × 2 storage profiles. Eight methods × 256 held-out traces = 98,304 evaluations. Planning uses 32 separate traces. V3 uses a fresh seed, so these are not v2's held-out traces.

Checkpoint intervals are 20, 40, 80 and 240 committed steps. With 240 steps total, interval 240 means no periodic checkpoint before completion; the initial durable version remains available. No final output checkpoint is charged to any method.

The joint search scores up to 32 placement/protection layouts × 4 intervals = 128 plans. Failure-aware and failure-unaware selectors see the same completed plans. The fixed-80 control sees the same layouts at interval 80. A whole layout block must finish before any selector can use it. The three selectors share a 30-second cap per case; candidate counts are recorded.

Profiles differ only in durable-storage speed (2 versus 0.25 GiB/s). Budgets are 1.75× reference time and 1.5× reference cost, including each profile's reference checkpoint overhead. Failure rates, replacement time and reinitialization costs are unchanged. These remain synthetic assumptions, not calibrated hardware parameters.


In [ ]:
RUN_NAME = "pilot_v3"
MAX_MINUTES = 30

CONFIG = read_json(ROOT / "configs" / "pilot_v3.json")
CONFIG["active_nodes"] = [4, 8, 16]  # 1, 2 or 4 groups of 4 shards.
CONFIG["regimes"] = ["homogeneous", "cheap_risky", "fast_risky", "correlated"]
CONFIG["instances_per_regime"] = 2
CONFIG["planning_traces"] = 32
CONFIG["evaluation_traces"] = 256
CONFIG["checkpoint_intervals"] = [20, 40, 80, 240]
CONFIG["checkpoint_layouts"] = 32
CONFIG["checkpoint_seconds"] = 30.0
CONFIG["workload"]["checkpoint_every"] = 80  # Reference policy; keep in the grid above.
CONFIG["workload"]["backup_every"] = 20     # Fixed across all methods.
CONFIG["workload"]["backup_budget_gib"] = 8.0
CONFIG["workload"]["required_success"] = 0.95

OUTPUT = ROOT / "results" / RUN_NAME
print("Output:", OUTPUT)
print(json.dumps(CONFIG, indent=2))


## Inspect one generated case

No simulation runs in this cell. Machine prices, speeds, risk and bandwidth are generated in `learnedplacement/model.py`. Here you can see the actual deadlines and cost budgets before starting.


In [ ]:
cases = make_cases(CONFIG)  # Input generation only; no simulations.
print("Cases:", len(cases), "Methods:", len(CONFIG["methods"]))
print("Held-out evaluations:", len(cases) * len(CONFIG["methods"]) * CONFIG["evaluation_traces"])
for example in cases[:len(CONFIG["profiles"])]:
    print(example.case_id,
          "deadline:", round(example.deadline_s, 2),
          "cost budget:", round(example.cost_budget_usd, 4),
          "storage GiB/s:", example.storage_gib_s,
          "replacement s:", example.replacement_s)
print(json.dumps([vars(n) for n in cases[0].nodes], indent=2))


## Run / resume

This cell starts the experiment. It uses one CPU process. Completed cases/plans and batches of observations save locally. Interrupting the cell is supported. A killed kernel may leave `.run.lock`; remove that file only after confirming the old process has stopped.

The exact source/configuration fingerprint is checked on resume. If you pull changed core code or edit the configuration, use a new run name. No completed result is overwritten by a different experiment.


In [ ]:
run_status = run(config=CONFIG, output=OUTPUT, max_minutes=MAX_MINUTES)
print(json.dumps(run_status, indent=2))


## Progress and results table

You can run these cells after a pause. They only read saved results. `statistical_screen_passed` is a per-plan Monte Carlo screen, not a real-world guarantee. Censored traces have no reported mean completion time or total cost.


In [ ]:
print(json.dumps(status(OUTPUT), indent=2))

import pandas as pd
from IPython.display import display, FileLink

summary_path = OUTPUT / "summary.csv"
if summary_path.exists():
    results = pd.read_csv(summary_path)
    display(results[["case_id", "profile", "method", "checkpoint_every", "backup_gib",
                     "mean_completion_s", "failure_free_completion_s", "failure_penalty_s",
                     "mean_cost_usd", "deadline_success", "statistical_screen_passed",
                     "censored_traces", "planning_s", "candidates", "completed_layouts"]])
    print("Selected checkpoint intervals (counts):")
    display(pd.crosstab(results.method, results.checkpoint_every))
else:
    print("No completed comparisons yet. Resume the run above.")


## Plots from saved results

The time/cost reference is the cheap speed-grouped placement with its checkpoint interval tuned on planning traces. This is a stronger comparison than a fixed-interval baseline.

The diagnostic plot splits the total time difference into failure-free time difference and additional time caused by failures. A gain from reducing checkpoint overhead alone is not evidence of better failure handling. Inspect feasibility, lost work and chosen intervals too. Censored cases stay in the table; decomposition requires uncensored pairs. Ablations inherit search and are excluded from planning-latency plots. Selector latencies are attributed scoring costs, not independently timed end-to-end planners.


In [ ]:
from learnedplacement.plots import plot_results, plot_failure_effects

ACTIVE_NODES_TO_PLOT = 4  # Or 8, 16.
PROFILE_TO_PLOT = "reference"  # Or "slow_storage".
table = plot_results(OUTPUT, active_nodes=ACTIVE_NODES_TO_PLOT,
                     profile=PROFILE_TO_PLOT, reference_method="speed_grouped_tuned")
if table is not None:
    display(table)
diagnostics = plot_failure_effects(OUTPUT, active_nodes=ACTIVE_NODES_TO_PLOT,
                                   profile=PROFILE_TO_PLOT,
                                   selected="checkpoint_joint", baseline="speed_grouped_tuned")
if diagnostics is not None:
    display(diagnostics)


## Paired comparisons

These comparisons answer different questions. Against `speed_grouped_tuned` and `domain_packed_tuned`: does joint search beat cheap placements with tuned checkpoints? Against `checkpoint_fixed_search`: does allowing checkpoint choice help when layout exploration is held constant? Against `checkpoint_unaware`: does scoring failures help over the identical full candidate pool? The last two ablations hold the joint plan's other decisions exactly fixed.

Negative differences favor joint search. Read both feasibility screens and actual candidate counts. A zero-layout fallback is not an optimized plan. Bootstrap intervals reflect simulation variation within a case, not model error or simultaneous guarantees. These held-out results never select plans.


In [ ]:
for baseline in ["speed_grouped_tuned", "domain_packed_tuned", "checkpoint_fixed_search",
                 "checkpoint_unaware", "checkpoint_fixed_ablation", "checkpoint_no_backup"]:
    print("checkpoint_joint vs", baseline)
    comparisons = paired_report(OUTPUT, selected="checkpoint_joint", baseline=baseline)
    display(pd.DataFrame(comparisons))


## Compact export for sharing

Stop or pause the runner first. The default export contains only analysis data: configuration/source hashes, case inputs, selected plans, summaries, failure-free measurements and paired statistics for every available method pair. It automatically computes the comparisons from saved observations; running the plotting/comparison cells first is optional.

Detailed traces, full candidate-search logs, plots and source copies stay local. Nothing is uploaded automatically. This export reads saved results and does not run simulations. It works for incomplete runs too, with shared-prefix statistics labeled as partial. Approximate paired mean intervals here use a normal approximation, not the bootstrap used in the comparison cell.

After pulling this export update, restart the kernel and run only setup, settings and this export cell to share an existing completed run. Do not rerun the experiment. A full troubleshooting archive is optional: `export_run(OUTPUT, full=True)`.


In [ ]:
from IPython.display import display, FileLink
archive = export_run(OUTPUT)  # Compact analysis export by default.
print("Share this ZIP:", archive)
print("Size:", round(archive.stat().st_size / 1024, 1), "KiB")
import os
display(FileLink(os.path.relpath(archive, Path.cwd())))
